In [10]:
import os, sys, pickle, importlib, gc
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.abspath('../../'))
import src.logit_graph.simulation
sys.modules['src.simulation'] = src.logit_graph.simulation

In [11]:
DATASETS = {
    'Facebook':    'runs/fitted_graphs_comparison_facebook5',
    'Twitter':     'runs/fitted_graphs_comparison_twitter2',
    'Google Plus': 'runs/fitted_graphs_comparison_gplus1',
    'Reddit':      'runs/fitted_graphs_comparison_reddit3',
    'Twitch':      'runs/fitted_graphs_comparison_twitch',
}

In [12]:
def load_summary_dfs(folder):
    """Load all comparator summary DataFrames from a runs folder.

    Each pkl file contains a growing list of GraphModelComparator objects.
    The last element in each file is the comparator for the graph named in
    the filename, so we load every file and take its final element.
    After collecting all summary DataFrames we de-duplicate by graph_filename
    (keeping the last occurrence) to guard against overlapping saves.
    """
    pkl_files = sorted(
        [f for f in os.listdir(folder)
         if f.endswith('.pkl') and 'comparators' in f.lower()]
    )
    if not pkl_files:
        print(f'  No comparator pkl files found in {folder}')
        return []

    dfs = []
    for fname in pkl_files:
        path = os.path.join(folder, fname)
        with open(path, 'rb') as fh:
            data = pickle.load(fh)

        if isinstance(data, list) and len(data) > 0:
            comp = data[-1]
        else:
            comp = data

        if hasattr(comp, 'summary_df') and comp.summary_df is not None:
            dfs.append(comp.summary_df)

        del data, comp
        gc.collect()

    seen = {}
    for df in dfs:
        gname = df['graph_filename'].iloc[0]
        seen[gname] = df
    return list(seen.values())

In [ ]:
from tqdm import tqdm
rows = []

for dataset_name, folder in DATASETS.items():
    print(f'Loading {dataset_name} from {folder} ...')
    if not os.path.isdir(folder):
        print(f'  ⚠ Folder not found — skipping')
        continue

    summary_dfs = load_summary_dfs(folder)
    print(f'  Found {len(summary_dfs)} graphs')

    for sdf in tqdm(summary_dfs, desc='Processing graphs'):
        graph_id = sdf['graph_filename'].iloc[0].replace('.edges', '')
        lg_row = sdf[sdf['model'] == 'LG']
        gic = lg_row['gic_value'].iloc[0] if not lg_row.empty else np.nan
        rows.append({'dataset': dataset_name, 'graph_id': graph_id, 'gic': gic})

results_df = pd.DataFrame(rows)
print(f'\nTotal graphs: {len(results_df)}')
results_df

Loading Facebook from runs/fitted_graphs_comparison_facebook5 ...
  Found 10 graphs


[  graph_filename     model  gic_value               param  fit_success  nodes  \
 0        0.edges  Original        NaN                 N/A         True    333   
 1        0.edges        LG   1.093779  d=2, sigma=-5.5510         True    333   
 2        0.edges        WS   0.838123                20.2         True    333   
 3        0.edges        BA   1.179342                 5.0         True    333   
 4        0.edges        ER   2.940206            0.094444         True    333   
 5        0.edges       GRG   8.230977                 1.0         True    333   
 
    edges   density  avg_clustering  avg_path_length  diameter  assortativity  \
 0   2519  0.045570        0.508245         3.752742        11       0.236039   
 1   2725  0.049296        0.050736         2.378758         4      -0.026064   
 2   2664  0.048193        0.046780         2.396668         4      -0.040401   
 3   1640  0.029668        0.103113         2.617841         4      -0.098849   
 4   5239  0.094775

Processing graphs: 100%|██████████| 10/10 [00:00<00:00, 5858.78it/s]

Loading Twitter from runs/fitted_graphs_comparison_twitter2 ...


In [ ]:
results_df.to_csv('consolidated_gic_results.csv', index=False)
print('Saved to consolidated_gic_results.csv')